# Prepare data with a clear rule

## Objective

Use the existing preparation hook to admit products with enough training
history, keep an explanation for exclusions and validate configuration before reading data.
This standalone lesson uses invented data, a fresh kernel and Java 17.

Most users do not need a hook. Omit `prepare` when your data is already suitable.
A hook is for your application's decisions. AutoForecast still owns generic checks,
calendars, hashes and frozen storage. Direct split inputs deliberately do not accept hooks.

## Step 1: Start a temporary lesson
The next cell starts Spark. No external database is used.


In [ ]:
# Keep this lesson in a temporary folder and use two local Spark workers.
import datetime as dt
import tempfile
from pathlib import Path

from autoforecast import StudyRegistry, configure_logging, register_study
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

workspace = tempfile.TemporaryDirectory(prefix="autoforecast-lesson-")
work = Path(workspace.name)
spark = (SparkSession.builder.master("local[2]")
         .config("spark.sql.shuffle.partitions", "2")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
configure_logging()


### Understand the output

Spark is ready.

## Step 2: Make observations
A has five training rows; B has four. Later rows are held out for evaluation.


In [ ]:
# Make two products. Product B starts one day later than A.
observations = spark.createDataFrame([
    (product, dt.date(2024, 1, day), float(day))
    for product in ("A", "B")
    for day in range(1 if product == "A" else 2, 10)
], "product string, date date, units double")
observations.orderBy("product", "date").show()


### Understand the output

The displayed table is the full input, including later actuals. Eligibility must use
only the permitted training portion.

## Step 3: Open a registry
This minimal file configures storage only. It is separate from the full study definition.


In [ ]:
# A registry file says where to save studies. It does not describe the data.
import yaml

registry_path = work / "registry.yaml"
registry_path.write_text(yaml.safe_dump({"storage": {"root": str(work / "registry")}}))
registry = StudyRegistry.from_config(registry_path, spark=spark)
print(registry.root)


### Understand the output

The printed directory is where study files will go.

## Step 4: Describe the source and dates
The full configuration below uses Parquet. Human-authored YAML and JSON are both accepted.


In [ ]:
# Save input data and a complete configured-source registration file.
from autoforecast import validate_study_config

observations.write.parquet(str(work / "observations"))
configuration = {
    "schema_version": 1,
    "dataset": {"schema_version": 1, "dataset": {"id": "example", "panel": {
        "series_col": "product", "time_col": "date", "target_col": "units",
        "frequency": "D", "season_length": 1, "target_estimand": "observed_nonnegative",
    }}},
    "recipe": {
        "schema_version": 2, "source_keys": ["product"], "training_start": "2024-01-01",
        "validation_origins": ["2024-01-05"], "validation_horizon": 2,
        "final_train_cutoff": "2024-01-07", "final_test_horizon": 2,
        "history_policy": "fixed", "new_series_policy": "error", "target_unit": "units",
        "feature_availability": {}, "metric": "mae",
        "history_missing": "preserve", "actual_missing": "error",
    },
    "source": {"kind": "parquet", "path": "observations"},
    "storage": {"root": str(work / "registry")},
    "preparation": {"minimum_rows": 4},
}
config_path = work / "study.yaml"
config_path.write_text(yaml.safe_dump(configuration))

# Check local settings. This does not read the source or create registry tables.

checked = validate_study_config(config_path)
print("Checked:", checked.checked)
print("Not checked:", checked.not_checked)


### Understand the output

The validator reports field, scientific and storage checks. This Parquet example has no
SQL to check. A Delta configuration also reports its local SQL syntax check.
It does not prove credentials, source existence, data quality or write permissions.
The machine-readable contract is available from `study_config_schema()` and in the package's
`study_config.schema.json`. Unknown scientific fields are rejected. Some checks need more
than the schema, such as calendar arithmetic and whether two effective table names are
the same. `validate_study_config` performs these local checks too.

## Step 5: Write the preparation rule
The class below needs no base class. Both `validate(settings)` and `prepare(context)` must
be callable. The first checks settings only. The second receives captured data.
Spark DataFrames are lazy: filtering and joining build a plan that runs later.
Keep these operations in Spark. Do not collect all rows onto the notebook computer just
to implement a preparation hook.


In [ ]:
# Keep business eligibility in the application, using training history only.
from autoforecast import PreparationResult
from autoforecast.study_registration import EXCLUSION_REASON_COLUMN


class EnoughHistory:
    def validate(self, settings):
        # Check the declared rule before any source data is read.
        minimum = settings.parameters.get("minimum_rows")
        if type(minimum) is not int or minimum < 1:
            raise ValueError("minimum_rows must be a positive integer")

    def prepare(self, context):
        # Count only the training rows allowed by this saved recipe.
        recipe = context.settings.recipe
        cutoff = recipe["validation_origins"][0]
        history = context.observations.filter(F.col("date").between(
            dt.date.fromisoformat(recipe["training_start"]), dt.date.fromisoformat(cutoff)))
        counts = history.groupBy("product").count()
        enough = counts.filter(F.col("count") >= context.settings.parameters["minimum_rows"])
        retained = context.starting_keys.join(enough.select("product"), ["product"], "inner")

        # The anti-join keeps the correct key types even when no product is excluded.
        excluded = context.starting_keys.join(retained, ["product"], "left_anti")
        excluded = excluded.withColumn(EXCLUSION_REASON_COLUMN, F.lit("too few training rows"))

        # Keep every observation for the retained products, including later actuals.
        return PreparationResult(
            observations=context.observations.join(retained, ["product"], "inner"),
            starting_keys=context.starting_keys, retained_keys=retained, exclusions=excluded,
            identifier="enough-training-history", version="1",
            policy={"minimum_rows": context.settings.parameters["minimum_rows"]},
            actions=("applied the declared training-history rule",),
        )


### Understand the output

`prepare` returns the original population, the retained products and the excluded products
with a reason. Filtering all observations by the retained keys keeps stage membership
consistent. The version describes this rule's behavior, so change it when the rule changes.

The context fields are:

| Field | What it contains |
|---|---|
| `settings.dataset`, `.recipe`, `.parameters` | Captured columns, dates/policies and your scalar settings |
| `observations` | Main captured input DataFrame |
| `audits` | Named captured queries, if configured |
| `starting_keys` | One row per original key; the denominator before exclusions |
| `findings` | Generic checks already found by the library |
| `scratch_directory` | Temporary directory owned by this attempt |

Key frames contain exactly the configured key columns, with the same types and no nulls
or duplicates. Exclusions add the nonblank string `EXCLUSION_REASON_COLUMN`.
Observations use the configured numeric target and canonical dates (or timezone-naive
hourly timestamps). Names starting `__af_` are reserved; use only the named contract fields.

Every result must include:

| Field | What to return |
|---|---|
| `observations` | Prepared observation rows for the retained keys |
| `starting_keys` | The original key frame, unchanged |
| `retained_keys` | One row for each key kept by the rule |
| `exclusions` | Every excluded key and its nonblank reason; keep the types even when empty |
| `identifier` | A stable, nonblank name for your rule |
| `version` | A nonblank version; change it when the rule's behavior changes |
| `policy` | Named decisions using strings, booleans, finite numbers or `None`; no nested objects |

Optional result fields:

| Field | When to use it |
|---|---|
| `actions` | Describe changes actually applied |
| `planned_actions` | Describe intentions that have not been applied |
| `findings` | Add factual findings with exactly `context.findings.schema` |
| `starts` | Declare a start for each key using string `START_COLUMN` |
| `reference_input` | Supply history for the configured reference calculator |
| `scoring_references` | Supply keys, stage, origin, weight and scale when `references.prepared` is true |

`actions` and `planned_actions` are tuples of nonblank strings. Reporting an action does
not apply it: the returned observations must already contain any stated change.
`reference_input` uses the configured key, date, numeric target and numeric weight columns.
It supplies history to the existing calculator, which uses only the permitted history for
each boundary. Do not return it together with `scoring_references`.

Precomputed `scoring_references` have exactly `keys`, `stage`, `origin`, `weight`, `scale`.
`keys` is a struct with the configured key names and types, without null key values.
`stage` is `tuning`, `validation` or `test`; `origin` is its canonical ISO string.
`weight` and `scale` are numeric, checked by the existing reference rules. Supply one row
per retained key and configured scoring boundary. Do not create rows for absent validation.
The WRMSSE example below shows the simpler calculator route.

Findings have `keys` (a struct with the configured key types), string fields `stage`,
`origin`, `time`, `column`, `issue_code`, `observed_value`, `message`, `status`, and
`affected_rows` (long integer). Use `context.findings.schema` to keep its exact nullability.
Context fields such as stage/origin/time may be null when they do not apply. Status must
be non-null `passed`, `warning` or `failed`; affected_rows must be non-null and nonnegative.
A failed finding blocks registration. Do not label a planned correction as an applied one.

## Step 6: Keep both products
A minimum of four training rows admits A and B. The exclusions DataFrame is empty but typed.


In [ ]:
# Run the declared rule and check the retained population.
first_id = register_study(config_path, registry=registry, prepare=EnoughHistory())
with registry.load_study(first_id) as study:
    print("Products retained:", study.n_series)
    assert study.n_series == 2


### Understand the output

The result is 2. Nothing was silently dropped.

## Step 7: Make the rule stricter
Now require five training rows. Only B is excluded, with the reason from the hook.


In [ ]:
# A changed rule creates a new study, leaving the first study intact.
configuration["preparation"]["minimum_rows"] = 5
config_path.write_text(yaml.safe_dump(configuration))
second_id = register_study(config_path, registry=registry, prepare=EnoughHistory())
with registry.load_study(second_id) as study:
    print("Products retained:", study.n_series)
    assert study.n_series == 1
assert first_id != second_id


### Understand the output

The result is 1. The saved admission evidence records B and its reason.

## Step 8: Check that a missing product cannot disappear silently

The library checks the preparation result against the starting population. The next example deliberately removes B's observations without declaring an exclusion. It should fail. This is a test of an incorrect hook, not a pattern to use in your application.


In [ ]:
# Make a deliberately incorrect hook and confirm that registration rejects it.
from dataclasses import replace


class HiddenLoss(EnoughHistory):
    def prepare(self, context):
        result = super().prepare(context)
        return replace(result, observations=result.observations.filter(F.col("product") == "A"))

# Use the rule that should retain both products, so B's unexplained loss is visible.
configuration["preparation"]["minimum_rows"] = 4
config_path.write_text(yaml.safe_dump(configuration))
try:
    register_study(config_path, registry=registry, prepare=HiddenLoss())
except ValueError as error:
    print("Rejected hidden product loss:", error)
else:
    raise AssertionError("An unexplained missing product was accepted")


### Understand the output

The error means the observations no longer match the declared retained products. A valid hook must record an exclusion and its reason, as `EnoughHistory` does.

## Step 9: Check that saved evidence cannot be changed

A saved study must keep its original evidence. Below we temporarily alter a file in this disposable lesson, try to load the study, and restore the original bytes in `finally`. Never edit a real study to correct data. Register the corrected data as a new study instead.


In [ ]:
# Change only the temporary lesson's saved preparation evidence.
definition, _ = registry.definition(first_id)
evidence = registry.root / definition["admission_evidence"]["path"]
original = evidence.read_bytes()
try:
    evidence.write_bytes(original + b"\n")
    try:
        with registry.load_study(first_id):
            pass
    except ValueError as error:
        print("Rejected changed saved evidence:", error)
    else:
        raise AssertionError("Changed study evidence was accepted")
finally:
    # Restore the exact bytes so the remaining lesson can continue safely.
    evidence.write_bytes(original)


### Understand the output

The hash check catches even a change that leaves the JSON readable. Restoring the exact saved bytes makes this temporary study usable again. In normal work, keep published studies unchanged and register a new version when the inputs or rules change.


The library rejects unreported ID loss, changed captured inputs, duplicate keys and
inconsistent retained/excluded sets. Starting and retained counts must reconcile.

## Managed Delta source
On a Delta-capable session, keep credentials outside this file. Replace `source` with a
managed SELECT file and declared table aliases. For example:

```yaml
source:
  kind: delta
  sql: observations.sql
  tables:
    observations: catalog.schema.observations
storage:
  root: /Volumes/catalog/schema/studies
  namespace: catalog.schema
```

The local `observations.sql` file contains `SELECT product, date, units FROM observations`
with any explicit date conditions you need. Offline validation parses it without running it.
Registration pins source versions and uses the existing shared registry tables.
`namespace` is optional; table overrides require it. Relative paths resolve beside the
configuration file. `StudyRegistry.from_config` uses file settings; registration without an
explicit registry and `from_environment` also apply the documented environment overrides.

## WRMSSE references
MAE needs no extra references. WRMSSE uses weights and scales from permitted history.
For an input that also has a numeric `revenue` column, the extra configuration is:

```yaml
recipe:
  # Keep the other recipe fields from the complete example above.
  metric: wrmsse
references:
  weight_col: revenue
  missing_history: na
  duplicate_policy: error
  weight_window_days: 3
  scale_lag: 1
  scale_start: history_start
```

Use these same reference settings with the direct API's `references` argument. Testing
values never supply selection weights or scales. There are no validation references when
validation is omitted. Existing zero-scale and missing-coverage checks still apply.
Generated registration records are JSON by default; `records: {format: yaml}` selects YAML.

## Step 10: Try WRMSSE with complete settings
The next cell copies our full configuration, adds a revenue column and registers the
weight/scale calculation. No application exclusion rule is applied in this example.


In [ ]:
# Add a synthetic weight input and keep the full study definition from above.
import copy

weighted = observations.withColumn("revenue", F.col("units") * 2)
weighted.write.parquet(str(work / "weighted-observations"))
weighted_config = copy.deepcopy(configuration)
weighted_config.pop("preparation")
weighted_config["source"]["path"] = "weighted-observations"
weighted_config["recipe"]["metric"] = "wrmsse"
weighted_config["references"] = {
    "weight_col": "revenue", "missing_history": "na", "duplicate_policy": "error",
    "weight_window_days": 3, "scale_lag": 1, "scale_start": "history_start",
}
weighted_path = work / "weighted.yaml"
weighted_path.write_text(yaml.safe_dump(weighted_config))
validate_study_config(weighted_path)
weighted_id = register_study(weighted_path, registry=registry)
with registry.load_study(weighted_id) as study:
    study.scoring_references.show()


### Understand the output

Each reference row names a series and a frozen scoring boundary. Validation uses its
allowed training history; testing uses the later final-fit history. Weights and scales
come from the calculator, not from testing targets.

## Step 11: Prepare a managed Delta configuration
This function writes a complete local config for an existing table with the same columns.
It validates the SQL locally. It does not query or write to the database. Call the function
with your own table name, then pass its returned path to `register_study` on a Delta-capable
Spark session when you are ready.


In [ ]:
# Reuse the complete scientific settings and change only the source route.
def delta_configuration(table_name, directory):
    import copy
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    value = copy.deepcopy(configuration)
    value.pop("preparation")
    value["source"] = {
        "kind": "delta", "sql": "observations.sql", "tables": {"observations": table_name},
    }
    value["storage"] = {"root": str(directory / "registry")}
    (directory / "observations.sql").write_text("SELECT product, date, units FROM observations")
    path = directory / "study.yaml"
    path.write_text(yaml.safe_dump(value))
    validate_study_config(path)
    return path

# Demonstrate the file checks with a generic table name. No table lookup occurs.
print(delta_configuration("catalog.schema.observations", work / "delta-example"))


### Understand the output

The printed file is ready for live source verification on your configured Delta runtime.
For shared Delta registry tables, add the existing `storage.namespace` setting.

## Step 12: Finish
The next cell closes Spark and removes only this lesson's temporary files.


In [ ]:
# Close Spark before removing the lesson's temporary files.
spark.stop()
workspace.cleanup()


### Understand the output

Use a durable registry for real studies. Keep application eligibility rules in your application.
